# FinSynth Ledger Lab

Block 1: dataset schema registry.


In [ ]:
# Imports
import json
import pandas as pd
from openai import OpenAI
from huggingface_hub import InferenceClient, login
from dotenv import load_dotenv

load_dotenv()


True

In [21]:
# Create the openAI client for later use
client = OpenAI()
# login(os.environ.get("HF_TOKEN"), add_to_git_credential=True)
hf_client = InferenceClient()

In [22]:
DATASET_PRESETS = {
    "personal_bank_statement": {
        "label": "Personal bank statement",
        "fields": [
            "date",
            "working",
            "description",
            "withdrawal",
            "deposit",
            "balance",
        ],
        "categories": [
            "salary",
            "groceries",
            "food",
            "transport",
            "subscriptions",
            "transfers",
            "refunds",
        ],
        "rules": [
            "Return JSON only.",
            "Use fake synthetic financial activity only.",
            "Keep the running balance internally consistent.",
        ],
    },
    "credit_card_spending_log": {
        "label": "Credit card spending log",
        "fields": ["date", "description", "amount_spent"],
        "categories": [
            "restaurants",
            "transport",
            "subscriptions",
            "online shopping",
            "groceries",
        ],
        "rules": [
            "Return JSON only.",
            "Use fake synthetic spending data only.",
            "Amounts must be numeric.",
        ],
    },
    "company_expense_ledger": {
        "label": "Company expense ledger",
        "fields": ["date", "description", "amount_spent"],
        "categories": ["SaaS", "office supplies", "travel", "meals", "vendor payments"],
        "rules": [
            "Return JSON only.",
            "Use fake company or vendor style data only.",
            "Descriptions should read like business expenses.",
        ],
    },
}


def get_dataset_preset(dataset_type):
    preset = DATASET_PRESETS.get(dataset_type)
    if preset is None:
        raise ValueError(f"Unknown dataset_type: {dataset_type}")
    return preset


In [23]:
get_dataset_preset("personal_bank_statement")

{'label': 'Personal bank statement',
 'fields': ['date',
  'working',
  'description',
  'withdrawal',
  'deposit',
  'balance'],
 'categories': ['salary',
  'groceries',
  'food',
  'transport',
  'subscriptions',
  'transfers',
  'refunds'],
 'rules': ['Return JSON only.',
  'Use fake synthetic financial activity only.',
  'Keep the running balance internally consistent.']}

In [24]:
def build_dataset_prompt(dataset_type, row_count):
    if row_count <= 0:
        raise ValueError("row_count must be positive")

    preset = get_dataset_preset(dataset_type)
    fields = ", ".join(preset["fields"])
    categories = ", ".join(preset["categories"])
    rules = "\n".join(f"- {rule}" for rule in preset["rules"])

    return (
        f"Generate exactly {row_count} rows of synthetic {preset['label'].lower()} data.\n"
        "Return JSON only as a list of objects.\n"
        f"Required fields: {fields}.\n"
        f"Include realistic variety from: {categories}.\n"
        f"Rules:\n{rules}\n"
        "Do not include real personal data, real account numbers, or explanations."
    )


In [25]:
def validate_dataset_rows(dataset_type, rows):
    """Act as a guardrail to further check the json generated after chatGPT api finish polishing the dataset."""
    preset = get_dataset_preset(dataset_type)
    expected_fields = preset["fields"]

    if not isinstance(rows, list):
        return False, "Rows must be a list of objects"

    for index, row in enumerate(rows):
        if not isinstance(row, dict):
            return False, f"Row {index} must be a dictionary"

        actual_fields = list(row.keys())
        if actual_fields != expected_fields:
            return False, f"Row {index} fields must match {expected_fields}"

        if dataset_type == "personal_bank_statement" and row["working"] not in {
            "Debit",
            "Credit",
        }:
            return False, f"Row {index} working must be Debit or Credit"

        if dataset_type != "personal_bank_statement":
            amount = row["amount_spent"]
            if not isinstance(amount, (int, float)):
                return False, f"Row {index} amount_spent must be numeric"

    return True, "OK"


In [26]:
def json_to_dataframe(clean_json, dataset_type=None):
    try:
        rows = json.loads(clean_json)
    except json.JSONDecodeError as exc:
        return None, f"JSON parsing failed: {exc.msg}"

    if dataset_type is not None:
        is_valid, message = validate_dataset_rows(dataset_type, rows)
        if not is_valid:
            return None, message

    return pd.DataFrame(rows), None


def save_csv(df, filename="synthetic_finance_data.csv"):
    if df is None or df.empty:
        return None, "DataFrame is empty"

    df.to_csv(filename, index=False)
    return filename, None


## Defining prompts for ChatGPT (for refining of JSON)


In [27]:
def repair_json_with_gpt(raw_text, dataset_type, model="gpt-4.1-mini"):
    preset = get_dataset_preset(dataset_type)
    schema_text = ", ".join(preset["fields"])
    rules_text = " ".join(preset["rules"])

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "system",
                "content": "Repair synthetic finance data into valid JSON only.",
            },
            {
                "role": "user",
                "content": (
                    f"Dataset type: {preset['label']}\n"
                    f"Required fields: {schema_text}\n"
                    f"Rules: {rules_text}\n"
                    "Return only parseable JSON as a list of objects.\n"
                    f"Raw model output:\n{raw_text}"
                ),
            },
        ],
    )
    return response.output_text.strip()


In [28]:
# result = repair_json_with_gpt(
#     [
#         {
#             "date": "2026-05-01",
#             "working": "Debit",
#             "description": "Salary",
#             "withdrawal": 0,
#             "deposit": 3000,
#             "balance": 3000,
#         }
#     ],
#     "personal_bank_statement",
#     model="gpt-4.1-mini",
# )

# print(result)

# # print("Block 5 tests passed")


In [29]:
def generate_raw_json(
    dataset_type,
    prompt,
    temperature,
    max_tokens,
    model="Qwen/Qwen2.5-72B-Instruct",
):
    preset = get_dataset_preset(dataset_type)
    schema_text = ", ".join(preset["fields"])
    rules_text = " ".join(preset["rules"])

    completion = hf_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    f"Generate synthetic {preset['label'].lower()} data. "
                    f"Required fields: {schema_text}. {rules_text}"
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content.strip()


In [30]:
# generate_raw_json(
#     "credit_card_spending_log",
#     build_dataset_prompt("credit_card_spending_log", 5),
#     0.4,
#     300,
# )


In [31]:
def run_generation_pipeline(
    dataset_type,
    row_count,
    temperature,
    repair_json=True,
    prompt_override=None,
    max_tokens=800,
    raw_generator=generate_raw_json,
    repairer=repair_json_with_gpt,
):
    prompt = prompt_override or build_dataset_prompt(dataset_type, row_count)
    raw_text = raw_generator(dataset_type, prompt, temperature, max_tokens)

    clean_json = raw_text
    if repair_json:
        clean_json = repairer(raw_text, dataset_type)

    df, warning = json_to_dataframe(clean_json, dataset_type)
    csv_path = None
    if warning is None:
        csv_path, warning = save_csv(df)

    return {
        "prompt": prompt,
        "raw_text": raw_text,
        "clean_json": clean_json,
        "dataframe": df,
        "csv_path": csv_path,
        "warning": warning,
    }


In [32]:
def get_default_prompt_for_preset(dataset_type, row_count=5):
    return build_dataset_prompt(dataset_type, row_count)


In [ ]:
# print(get_default_prompt_for_preset("credit_card_spending_log", row_count=3))
# print(get_default_prompt_for_preset("personal_bank_statement", row_count=10))


Generate exactly 3 rows of synthetic credit card spending log data.
Return JSON only as a list of objects.
Required fields: date, description, amount_spent.
Include realistic variety from: restaurants, transport, subscriptions, online shopping, groceries.
Rules:
- Return JSON only.
- Use fake synthetic spending data only.
- Amounts must be numeric.
Do not include real personal data, real account numbers, or explanations.
Generate exactly 10 rows of synthetic personal bank statement data.
Return JSON only as a list of objects.
Required fields: date, working, description, withdrawal, deposit, balance.
Include realistic variety from: salary, groceries, food, transport, subscriptions, transfers, refunds.
Rules:
- Return JSON only.
- Use fake synthetic financial activity only.
- Keep the running balance internally consistent.
Do not include real personal data, real account numbers, or explanations.
